In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 大作業 4: 氣象資料開放平臺 API 的單車景點一週日夜天氣預報網頁應用程式

## 大作業4-1：取得天氣預報數據
- 使用 CWA API 取得台灣各地區的一週天氣資料。
- 您必須使用傳回 JSON 格式資料的 API。
- 地區包括：
    - 北部地區
    - 中部地區
    - 南部地區
    - 東北部地區
    - 東部地區
    - 東南部地區

### 步驟一：安裝依賴

In [ ]:
%pip install -q requests streamlit streamlit_jupyter pandas plotly
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!rm cloudflared-linux-amd64.deb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 5.1 MB/s eta 0:00:00
--2026-09-18 03:43:51--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
L

### 步驟二：取得天氣預報數據

In [ ]:
import json
import requests

# Set the URL for the API endpoint
url = "https://opendata.cwa.gov.tw/fileapi/v1/opendataapi/F-B0053-009?Authorization=CWA-DCA1711B-56A9-452F-9CD8-C868B665AAEC&downloadType=WEB&format=JSON"

# Fetch the data, bypassing SSL verification for now
response = requests.get(url, verify=False)
data = response.json()

# Show the fetched data
# print(json.dumps(data, indent=4, ensure_ascii=False))

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opendata.cwa.gov.tw'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cwaopendata.s3.ap-northeast-1.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## 大作業 4-2：分析所取得的數據

### 步驟一：找出每個區域的每日最高溫度 (MaxT) 和最低溫度 (MinT)。

In [ ]:
# @title
# 擷取台灣各地區的一週天氣預報數據
temperature_forecasts = data["cwaopendata"]["Dataset"]["Locations"]["Location"]

# 找出每日最高氣溫 (MaxT) 和最低氣溫 (MinT)
for region in temperature_forecasts:
    region_name = region["LocationName"]
    print(f"地區：{region_name}")

    # 尋找 MaxT 與 MinT 元素
    maxt_element = None
    mint_element = None
    for element in region["WeatherElement"]:
        if element["ElementName"] == "最高溫度":
            maxt_element = element
        elif element["ElementName"] == "最低溫度":
            mint_element = element

    if maxt_element and mint_element:
        # 依日期對齊並印出最高與最低溫度
        for max_time, min_time in zip(maxt_element["Time"], mint_element["Time"]):
            date = max_time["StartTime"][:10]  # 取得日期部分 YYYY-MM-DD
            ampm = "白天" if max_time["EndTime"][11:13] == "18" else "晚上"
            max_temp = max_time["ElementValue"]["MaxTemperature"]
            min_temp = min_time["ElementValue"]["MinTemperature"]
            print(f"  日期: {date} | {ampm} | 最低氣溫: {min_temp}°C | 最高氣溫: {max_temp}°C")
    print("-" * 50)

地區：宜蘭河濱公園
  日期: 2026-09-18 | 白天 | 最低氣溫: 26°C | 最高氣溫: 29°C
  日期: 2026-09-18 | 晚上 | 最低氣溫: 23°C | 最高氣溫: 26°C
  日期: 2026-09-19 | 白天 | 最低氣溫: 23°C | 最高氣溫: 32°C
  日期: 2026-09-19 | 晚上 | 最低氣溫: 24°C | 最高氣溫: 28°C
  日期: 2026-09-20 | 白天 | 最低氣溫: 24°C | 最高氣溫: 30°C
  日期: 2026-09-20 | 晚上 | 最低氣溫: 22°C | 最高氣溫: 26°C
  日期: 2026-09-21 | 白天 | 最低氣溫: 22°C | 最高氣溫: 29°C
  日期: 2026-09-21 | 晚上 | 最低氣溫: 22°C | 最高氣溫: 26°C
  日期: 2026-09-22 | 白天 | 最低氣溫: 22°C | 最高氣溫: 29°C
  日期: 2026-09-22 | 晚上 | 最低氣溫: 22°C | 最高氣溫: 26°C
  日期: 2026-09-23 | 白天 | 最低氣溫: 22°C | 最高氣溫: 29°C
  日期: 2026-09-23 | 晚上 | 最低氣溫: 23°C | 最高氣溫: 26°C
  日期: 2026-09-24 | 白天 | 最低氣溫: 23°C | 最高氣溫: 29°C
  日期: 2026-09-24 | 晚上 | 最低氣溫: 24°C | 最高氣溫: 27°C
--------------------------------------------------
地區：員山公園
  日期: 2026-09-18 | 白天 | 最低氣溫: 26°C | 最高氣溫: 29°C
  日期: 2026-09-18 | 晚上 | 最低氣溫: 23°C | 最高氣溫: 26°C
  日期: 2026-09-19 | 白天 | 最低氣溫: 23°C | 最高氣溫: 32°C
  日期: 2026-09-19 | 晚上 | 最低氣溫: 24°C | 最高氣溫: 28°C
  日期: 2026-09-20 | 白天 | 最低氣溫: 24°C | 最高氣溫: 31°C
  日期: 2026-09-20 | 

## 大作業 4-3：將溫度資料儲存到 SQLite 資料庫
- 建立一個 SQLite 資料庫來儲存溫度資料。
- 該資料庫應包含一個以下列的表：
  - id：主鍵
  - regionName：區域名稱
  - dataDate：預報日期
  - MaxT：最高溫度
  - MinT：最低溫度

### 步驟一：將資料儲存到 SQLite 資料庫

In [ ]:
# @title
import sqlite3

# Connect to SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect("/content/drive/MyDrive/中興大學/人工智慧與資訊安全/colab/大作業④/data.db")
cursor = conn.cursor()

# Drop the old table if it exists
cursor.execute("DROP TABLE IF EXISTS TemperatureForecasts")

# Create a table to store the temperature forecasts
cursor.execute("""
CREATE TABLE TemperatureForecasts (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    regionName TEXT,
    date TEXT,
    ampm TEXT,
    maxt INT,
    mint INT
)
""")

# Insert data into the table
for region in temperature_forecasts:
    region_name = region["LocationName"]
    print(f"地區：{region_name}")

    maxt_element = None
    mint_element = None
    for element in region["WeatherElement"]:
        if element["ElementName"] == "最高溫度":
            maxt_element = element
        elif element["ElementName"] == "最低溫度":
            mint_element = element

    if maxt_element and mint_element:
        # 依日期對齊並印出最高與最低溫度
        for max_time, min_time in zip(maxt_element["Time"], mint_element["Time"]):
            date = max_time["StartTime"][:10]  # 取得日期部分 YYYY-MM-DD
            ampm = "白天" if max_time["EndTime"][11:13] == "18" else "晚上"
            max_temp = int(max_time["ElementValue"]["MaxTemperature"])
            min_temp = int(min_time["ElementValue"]["MinTemperature"])
            # print(f"  日期: {date} | {ampm} | 最高氣溫: {max_temp}°C | 最低氣溫: {min_temp}°C")
            cursor.execute(
                "INSERT INTO TemperatureForecasts (regionName, date, ampm, maxt, mint) VALUES (?, ?, ?, ?, ?)",
                (region_name, date, ampm, max_temp, min_temp)
            )

# Commit changes and close the connection
conn.commit()
conn.close()

地區：宜蘭河濱公園
地區：員山公園
地區：宜蘭運動公園
地區：壯圍東港
地區：安農溪分洪堰風景區
地區：水源橋
地區：五濱路
地區：武荖坑
地區：七星潭
地區：南濱公園
地區：鯉魚潭
地區：馬太鞍社區
地區：瑞穗鄉
地區：石門洞
地區：烈嶼鄉_小金門車轍道
地區：金寧鄉_金寧線
地區：金沙鎮_金沙線
地區：金城鎮_金城線
地區：金湖鎮_金湖線
地區：貓羅溪
地區：日月潭
地區：綠色隧道
地區：集集火車站
地區：海神宮
地區：賽嘉樂園
地區：萬巒鄉
地區：關山
地區：後壁湖
地區：貓鼻頭
地區：貓貍山公園
地區：新屋綠色走廊
地區：龍潭大池
地區：頭寮陵寢
地區：梅鶴山莊
地區：桃園羅浮
地區：新竹馬武督山
地區：峨嵋湖
地區：環大湖社區
地區：旗山
地區：都會公園
地區：援中港
地區：龜山停車場
地區：北極殿
地區：愛河沿岸
地區：旗津
地區：高屏溪出海口
地區：四湖鄉海清宮
地區：成龍二橋
地區：萬里核二廠
地區：漁人碼頭
地區：淡水捷運站
地區：八里渡船頭
地區：十三行博物館
地區：捷運紅樹林站
地區：捷運竹圍站
地區：疏洪運動公園
地區：龍門公園
地區：永福橋_新店溪南岸
地區：浮州橋
地區：坪林茶葉博物館
地區：九芎根
地區：南寮漁港
地區：客雅溪口
地區：六腳鄉遊客中心
地區：朴子溪橋
地區：東石大橋
地區：彰化體育場
地區：護天宮
地區：橫山
地區：公園路
地區：田中森林公園仁愛之家_長青自行車道
地區：田中森林公園仁愛之家_長青自行車道(健腳級)
地區：松柏嶺
地區：二水車站
地區：高美溼地
地區：后豐鐵馬道
地區：神岡
地區：東勢
地區：豐原
地區：潭子
地區：臺中都會公園
地區：東海大學
地區：太原車站
地區：國美館
地區：臺中市楓樹古道
地區：中興大學
地區：中和禪寺
地區：大同公司
地區：關渡捷運站
地區：社子島
地區：劍潭捷運站_關渡自然公園
地區：劍潭捷運站_基隆河南北岸
地區：南港高工
地區：南港研究院路
地區：六張犁捷運站
地區：永福橋_淡水河東岸及新店溪北岸
地區：文山區捷運木柵站
地區：政大_木柵貓空
地區：政大_景美溪兩岸
地區：樟湖木柵觀光茶園
地區：小野柳風景區
地區：海濱公園
地區：臨海路
地區：長濱鄉
地區：萬安磚窯
地區：大坡池
地區：慈善堂
地區：關山環保公園
地區：臺東鹿野鄉永德
地區：龍田村
地區：綠島
地區：蘭嶼

### 步驟二: 檢查資料是否已正確保存。

In [ ]:
# @title
import pandas as pd

# Reconnect to the SQLite database
conn = sqlite3.connect("/content/drive/MyDrive/中興大學/人工智慧與資訊安全/colab/大作業④/data.db")

# Query to get all distinct region names
query = "SELECT DISTINCT regionName FROM TemperatureForecasts"
region_names = pd.read_sql_query(query, conn)

# Display the region names
print(region_names)

# Close the connection
conn.close()

          regionName
0             宜蘭河濱公園
1               員山公園
2             宜蘭運動公園
3               壯圍東港
4          安農溪分洪堰風景區
..               ...
113             七股潟湖
114           安平路運河岸
115             府平公園
116     澎湖縣_吉貝環島自行車道
117  澎湖縣_湖西北寮至龍門自行車道

[118 rows x 1 columns]


In [ ]:
# @title
# Reconnect to the SQLite database
conn = sqlite3.connect("/content/drive/MyDrive/中興大學/人工智慧與資訊安全/colab/大作業④/data.db")

# Load data into a pandas DataFrame
selected_region = "宜蘭河濱公園"
query = f"SELECT regionName, date, ampm, maxt, mint FROM TemperatureForecasts WHERE regionName = '{selected_region}'"

df_weather_from_db = pd.read_sql_query(query, conn)

# Display the DataFrame
print(df_weather_from_db)

# Close the connection
conn.close()

   regionName        date ampm  maxt  mint
0      宜蘭河濱公園  2026-09-18   白天    29    26
1      宜蘭河濱公園  2026-09-18   晚上    26    23
2      宜蘭河濱公園  2026-09-19   白天    32    23
3      宜蘭河濱公園  2026-09-19   晚上    28    24
4      宜蘭河濱公園  2026-09-20   白天    30    24
5      宜蘭河濱公園  2026-09-20   晚上    26    22
6      宜蘭河濱公園  2026-09-21   白天    29    22
7      宜蘭河濱公園  2026-09-21   晚上    26    22
8      宜蘭河濱公園  2026-09-22   白天    29    22
9      宜蘭河濱公園  2026-09-22   晚上    26    22
10     宜蘭河濱公園  2026-09-23   白天    29    22
11     宜蘭河濱公園  2026-09-23   晚上    26    23
12     宜蘭河濱公園  2026-09-24   白天    29    23
13     宜蘭河濱公園  2026-09-24   晚上    27    24


## 大作業 4-4: 實作溫度預報網頁應用程式（Streamlit）
- 建立一個 Streamlit 應用程式來視覺化溫度資料。
- 允許使用者從下拉式選單中選擇一個區域，並查看該區域的溫度預報。
- 此應用程式必須使用 SQL 從 SQLite 資料庫中查詢資料。
- 包含折線圖和表格，以顯示一週內的溫度資料。

In [ ]:
conn = sqlite3.connect("/content/drive/MyDrive/中興大學/人工智慧與資訊安全/colab/大作業④/data.db")

# Query to get all distinct region names
query = "SELECT DISTINCT regionName FROM TemperatureForecasts"
region_names = pd.read_sql_query(query, conn)

print(list(region_names["regionName"]))

['宜蘭河濱公園', '員山公園', '宜蘭運動公園', '壯圍東港', '安農溪分洪堰風景區', '水源橋', '五濱路', '武荖坑', '七星潭', '南濱公園', '鯉魚潭', '馬太鞍社區', '瑞穗鄉', '石門洞', '烈嶼鄉_小金門車轍道', '金寧鄉_金寧線', '金沙鎮_金沙線', '金城鎮_金城線', '金湖鎮_金湖線', '貓羅溪', '日月潭', '綠色隧道', '集集火車站', '海神宮', '賽嘉樂園', '萬巒鄉', '關山', '後壁湖', '貓鼻頭', '貓貍山公園', '新屋綠色走廊', '龍潭大池', '頭寮陵寢', '梅鶴山莊', '桃園羅浮', '新竹馬武督山', '峨嵋湖', '環大湖社區', '旗山', '都會公園', '援中港', '龜山停車場', '北極殿', '愛河沿岸', '旗津', '高屏溪出海口', '四湖鄉海清宮', '成龍二橋', '萬里核二廠', '漁人碼頭', '淡水捷運站', '八里渡船頭', '十三行博物館', '捷運紅樹林站', '捷運竹圍站', '疏洪運動公園', '龍門公園', '永福橋_新店溪南岸', '浮州橋', '坪林茶葉博物館', '九芎根', '南寮漁港', '客雅溪口', '六腳鄉遊客中心', '朴子溪橋', '東石大橋', '彰化體育場', '護天宮', '橫山', '公園路', '田中森林公園仁愛之家_長青自行車道', '田中森林公園仁愛之家_長青自行車道(健腳級)', '松柏嶺', '二水車站', '高美溼地', '后豐鐵馬道', '神岡', '東勢', '豐原', '潭子', '臺中都會公園', '東海大學', '太原車站', '國美館', '臺中市楓樹古道', '中興大學', '中和禪寺', '大同公司', '關渡捷運站', '社子島', '劍潭捷運站_關渡自然公園', '劍潭捷運站_基隆河南北岸', '南港高工', '南港研究院路', '六張犁捷運站', '永福橋_淡水河東岸及新店溪北岸', '文山區捷運木柵站', '政大_木柵貓空', '政大_景美溪兩岸', '樟湖木柵觀光茶園', '小野柳風景區', '海濱公園', '臨海路', '長濱鄉', '萬安磚窯', '大坡池', '慈善堂', '關山環保公園', '臺東鹿野鄉永德', '龍田村', '綠島', '蘭嶼'

### 步驟一：建立 Streamlit 應用

In [ ]:
# @title
%%writefile /content/drive/MyDrive/中興大學/人工智慧與資訊安全/colab/大作業④/app.py

import streamlit as st
import sqlite3
import pandas as pd
import plotly.express as px


# Title of the app
st.title("單車景點一週日夜天氣預報網頁應用程式")

# Connect to the SQLite database
conn = sqlite3.connect("/content/drive/MyDrive/中興大學/人工智慧與資訊安全/colab/大作業④/data.db")

# Query to get all distinct region names
query = "SELECT DISTINCT regionName FROM TemperatureForecasts"
region_names = pd.read_sql_query(query, conn)

# Dropdown to select region
selected_region = st.selectbox("選擇一個區域", list(region_names["regionName"]))

# Query to get MaxT and MinT for the selected region
query = f"""
SELECT date, ampm, maxt, mint
FROM TemperatureForecasts
WHERE regionName = '{selected_region}'
"""
df_weather = pd.read_sql_query(query, conn)

# Close the connection
conn.close()

# Plot the data using Plotly
fig = px.line(
    df_weather,
    x="date",
    y=["maxt", "mint"],
    title=f"Temperature Trends for {selected_region}",
    labels={"value": "Temperature (°C)", "date": "Date"},
)
st.plotly_chart(fig)

# Display the data
st.write(f"Temperature Data for {selected_region}")
st.dataframe(df_weather)

Overwriting /content/drive/MyDrive/中興大學/人工智慧與資訊安全/colab/大作業④/app.py


### Step 2: Run the streamlit app

In [ ]:
!echo "Tunnel Password: $(curl -s ifconfig.io)"
!streamlit run /content/drive/MyDrive/中興大學/人工智慧與資訊安全/colab/大作業④/app.py &> /content/drive/MyDrive/中興大學/人工智慧與資訊安全/colab/大作業④/logs.txt &
!cloudflared tunnel --url http://localhost:8501


Tunnel Password: 34.59.185.7
2026-09-18T03:44:01Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-18T03:44:01Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-18T03:44:04Z INF +--------------------------------------------------------------------------------------------+
2026-09-18T03:44:04Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-18T03:44:04Z INF |  https://fleet-promotiona